<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/20_smoke_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 20 — Faz 2 smoke test'leri (SMK-001)

**Bilimsel amaç** (D-034 madde 10; `docs/PHASE2_DESIGN.md` Bölüm 5.1): gömme çıkarımına başlamadan önce her backbone için şunları göstermek:
- resmi ağırlık eksiksiz yükleniyor (`strict=True`, sha256);
- model resmi girdi biçimini alıyor (SR, frontend, `do_normalize`; `configs/model_input_contracts.yaml`);
- eval modunda deterministik ve batch'ten bağımsız (katılımcılar arası bilgi akışı yok);
- çıktı anlamlı: PANNs kelime kayıtlarında AudioSet "Speech" sınıfını üste koyuyor; bütün modellerde aynı kişinin kayıtları birbirine daha benzer;
- Faz 3 için: eğitim modunda batch 16 × 4 s, fp16, bir adımın GPU belleği ve süresi.

**Çalışma ortamı:** *Runtime → Change runtime type → T4 GPU*. Süre: ağırlık indirme ~10 dk (ilk sefer), test ~10 dk, smoke test ~10 dk.

**Önce yapılacak (bir kez, elle):** BEATs ağırlığı OneDrive'da; otomatik indirilemiyor. `github.com/microsoft/unilm/tree/master/beats` README tablosunda **"BEATs_iter3+ (AS2M)"** (fine-tuned *olmayan*, üçüncü sütun) bağlantısından indir ve Drive'da `MyDrive/asthma-voice/models/BEATs_iter3_plus_AS2M.pt` adıyla kaydet.

Herhangi bir backbone FAIL verirse **21 numaralı notebook'a geçme**; raporu birlikte inceleyelim.

## 0. Oturum kurulumu

In [ ]:
import subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
for p in [DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv', DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
for p in [MODELS, EXPERIMENTS, EMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Girdiler mevcut.')

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', url, str(REPO_DIR)], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
assert r.returncode == 0, 'git başarısız'
COMMIT = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Repo commit:', COMMIT)

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-colab.txt
import torch, torchaudio, transformers, torchlibrosa, sklearn, numpy as np
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| transformers', transformers.__version__,
      '| sklearn', sklearn.__version__, '| numpy', np.__version__)
import torchaudio.compliance.kaldi  # BEATs bunu gerektirir; yoksa burada hata verir (sözleşme notu)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Hat testi (ağırlıksız küçük modellerle, ~10 dk)
Gerçek veri ve gerçek ağırlık olmadan bütün kod yollarını sınar: pencereleme, kancalar, smoke test, gömme çıkarımı (kaldığı yerden devam dahil), problar (yerleştirilmiş sinyal bulunuyor, saf gürültü ≈ 0.5), DeLong / Holm / hızlı bootstrap. `Tüm kontroller geçti (P1–P9)` görmeden devam etme.

In [ ]:
!cd {REPO_DIR} && python tests/test_phase2.py

## 2. Ağırlıkları Drive'a indir (yalnız ilk sefer; sonra atlanır)
- **PANNs** (Zenodo 3987831, CC BY 4.0): dosya md5'i Zenodo'nun kendi kaydından okunup doğrulanır.
- **WavLM** (Hugging Face): çözümlenen commit hash'i `REVISION.txt`'ye yazılır; model ve özellik çıkarıcı `save_pretrained` ile Drive'a kopyalanır. Sonraki oturumlar internetteki sürüme değil bu kopyaya bakar.
- **BEATs:** elle (yukarıda). Burada yalnız varlığı kontrol edilir.

In [ ]:
import hashlib, json, requests
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

rec = requests.get('https://zenodo.org/api/records/3987831', timeout=60).json()
files = {f['key']: f for f in rec['files']}
for key in ['Cnn10_mAP=0.380.pth', 'Cnn14_mAP=0.431.pth', 'Cnn14_16k_mAP=0.438.pth']:
    dst = MODELS / key
    expected = files[key]['checksum'].split(':', 1)[1]
    if not dst.exists():
        url = files[key]['links']['self']
        print('indiriliyor:', key)
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            tmp = dst.with_suffix('.part')
            with open(tmp, 'wb') as f:
                for b in r.iter_content(1 << 22):
                    f.write(b)
        tmp.rename(dst)
    assert md5(dst) == expected, f'{key}: md5 Zenodo kaydıyla uyuşmuyor'
    print(key, 'md5 ✓', round(dst.stat().st_size / 1e6), 'MB')

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
for repo, name in [('microsoft/wavlm-base-plus', 'wavlm-base-plus'), ('microsoft/wavlm-large', 'wavlm-large')]:
    dst = MODELS / name
    if (dst / 'REVISION.txt').exists():
        print(name, 'mevcut, revizyon', (dst / 'REVISION.txt').read_text().strip()); continue
    rev = HfApi().model_info(repo).sha
    snap = snapshot_download(repo, revision=rev)
    model, li = WavLMModel.from_pretrained(snap, output_loading_info=True)
    fe = Wav2Vec2FeatureExtractor.from_pretrained(snap)
    assert not li['missing_keys'], li['missing_keys'][:5]
    model.save_pretrained(dst); fe.save_pretrained(dst)
    (dst / 'REVISION.txt').write_text(rev + '\n')
    (dst / 'download_info.json').write_text(json.dumps({'repo': repo, 'revision': rev,
        'unexpected_keys_at_download': li['unexpected_keys'], 'transformers': __import__('transformers').__version__}, indent=2))
    print(name, 'kaydedildi, revizyon', rev, '| indirmede beklenmeyen anahtar:', len(li['unexpected_keys']))

In [ ]:
beats = MODELS / 'BEATs_iter3_plus_AS2M.pt'
assert beats.exists(), ('BEATs ağırlığı yok. README tablosundaki "BEATs_iter3+ (AS2M)" (fine-tuned olmayan) bağlantısından indirip '
                        f'{beats} adıyla Drive\'a kaydet, sonra bu hücreyi yeniden çalıştır.')
print('BEATs mevcut,', round(beats.stat().st_size / 1e6), 'MB')

## 2b. Önbelleği yerel diske kopyala (sha256 doğrulamalı)

In [ ]:
import hashlib, shutil, json
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

info = json.loads((DRIVE_CACHE / 'cache_info.json').read_text())
LOCAL_CACHE.mkdir(exist_ok=True)
for name in ['audio_32k.f32', 'audio_16k.f32', 'index.csv']:
    if not (LOCAL_CACHE / name).exists():
        shutil.copy(DRIVE_CACHE / name, LOCAL_CACHE / name)
    assert sha256(LOCAL_CACHE / name) == info['files_sha256'][name], f'{name}: sha256 farklı'
print('Önbellek yerel diskte ve doğrulandı.')

## 3. SMK-001 (~10 dk)
Her backbone için S1–S13. Rapor `reports/phase2/SMK-001_smoke.md`. Checkpoint sha256'ları `models/MANIFEST.json`'a yazılır; sonraki oturumlarda değişirse S1 FAIL verir.

In [ ]:
!cd {REPO_DIR} && python scripts/smoke_test_backbones.py --cache-dir {LOCAL_CACHE} --model-dir {MODELS} --report-dir {REPO_DIR}/reports/phase2 --device cuda

## 4. Raporu GitHub'a gönder (ayrı dal)
Yalnız `reports/phase2/` commit edilir (agrega; katılımcı ID'si yok). Notebook'u kaydetmeden önce çıktıları temizle (*Edit → Clear all outputs*).

In [ ]:
BRANCH = 'phase2-smoke-results'
def git(*a):
    r = subprocess.run(['git', '-C', str(REPO_DIR), *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***').strip())
    return r
git('config', 'user.name', 'alperarslan19')
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
git('checkout', '-B', BRANCH)
git('add', 'reports/phase2')
staged = git('diff', '--cached', '--name-only').stdout.split()
assert staged and all(any(s.startswith(p) for p in ['reports/phase2']) for s in staged), f'Beklenmeyen dosya: {staged}'
assert git('commit', '-m', 'SMK-001: Faz 2 smoke test raporu (Colab)').returncode == 0
assert git('push', f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git', BRANCH).returncode == 0
print(f'PR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')